# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [1]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


NVIDIA RTX A4000, 16376 MiB, 16101 MiB
Deps installed


In [ ]:
!pip install "transformers==4.43.4" "sentence-transformers==2.7.0" "huggingface-hub==0.25.0" "accelerate==0.33.0" "peft==0.13.2" "bitsandbytes==0.43.3" -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cogmem 0.1.0 requires openai>=1.0, which is not installed.
cogmem 0.1.0 requires together>=1.0, which is not installed.
cogmem 0.1.0 requires peft>=0.7, but you have peft 0.6.2 which is incompatible.
cogmem 0.1.0 requires pyyaml>=6.0, but you have pyyaml 5.4.1 which is incompatible.


In [3]:
# Cell 2: Clone CogMem + load tasks
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || \
    (cd /notebooks/CogMem && git pull)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

# Load BigCodeBench full (1140 tasks)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print("Tasks:", len(tasks))
# Split: first 100 for patch creation, rest for evaluation
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


Already up to date.
Tasks: 1140
Train (create patches): 100
Eval (test patches): 1040


In [6]:
# Cell 3: Load base model (4-bit) + embedder
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading embedder...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. Free VRAM: {free:.1f} GB")
print("Ready for patch creation.")


Loading model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [3]:
# Cell 4: Create patches from first 100 tasks
# Generate N=4 candidates per task, find pass/fail contrasts, create patches
import time
from difflib import SequenceMatcher
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.patch import CognitivePatch
from cogmem.patches.bank import PatchBank
from cogmem.patches.create import create_patch_from_contrast
from cogmem.patches.wake import generate_with_model, find_best_contrast_pair

N_CANDIDATES = 4
PATCH_DIR = "/notebooks/cogmem_patches"
patch_bank = PatchBank(PATCH_DIR)

# Enable gradients on quantized model (run once before any patch creation)
from peft import prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)
print('Base model prepared for training')

total_patches = 0
total_passed = 0
start_time = time.time()

for i, task in enumerate(TRAIN_TASKS):
    task_id = task["task_id"]
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    # Generate N candidates
    candidates = []
    for _ in range(N_CANDIDATES):
        try:
            response = generate_with_model(
                base_model, tokenizer, messages, temperature=0.8
            )
            code = extract_code(response)
            if code and len(code.strip()) > 20:
                result = evaluate_solution(task, code, timeout=30, mode="subprocess")
                candidates.append({"code": code, "passed": result["passed"]})
        except Exception as e:
            if i < 3:  # only log first few
                print('  Gen error:', type(e).__name__, str(e)[:80])

    passes = [c for c in candidates if c["passed"]]
    fails = [c for c in candidates if not c["passed"]]

    if passes:
        total_passed += 1

    # Create patch if we have pass + fail
    if passes and fails:
        best_pair, sim = find_best_contrast_pair(passes, fails)
        if best_pair:
            try:
                patch = create_patch_from_contrast(
                    base_model, tokenizer, prompt,
                    best_pair["fail"]["code"],
                    best_pair["pass"]["code"],
                    patch_id="patch_{}_{}".format(task_id.replace("/", "_"), int(time.time())),
                    rank=2, n_steps=50, lr=5e-3,
                )
                patch.embedding = task_embedding
                patch.source_task_id = task_id
                patch_bank.add(patch)
                total_patches += 1
            except Exception as e:
                print("  Patch creation failed:", str(e)[:80])

    if (i + 1) % 10 == 0 or i < 5:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
        print("[{}/{}] {}: {}P/{}F | patches={} | pass_rate={}/{} | {:.0f}/hr".format(
            i + 1, len(TRAIN_TASKS), task_id,
            len(passes), len(fails), total_patches,
            total_passed, i + 1, rate))

patch_bank.save()
elapsed = (time.time() - start_time) / 60
print()
print("=" * 50)
print("PATCH CREATION COMPLETE")
print("Tasks processed:", len(TRAIN_TASKS))
print("Tasks with passes:", total_passed)
print("Patches created:", total_patches)
print("Time:", round(elapsed, 1), "min")
print("Bank stats:", patch_bank.stats())


2026-04-10 00:45:30.106687: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-10 00:45:30.106743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-10 00:45:30.107762: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-10 00:45:30.113245: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 00:45:30.973894: W tensorflow/compiler/tf2

Base model prepared for training
[2026-04-10 00:46:42,809] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


Step,Training Loss
5,2.327000


[1/100] BigCodeBench/0: 2P/2F | patches=1 | pass_rate=1/1 | 47/hr
[2/100] BigCodeBench/1: 4P/0F | patches=1 | pass_rate=2/2 | 63/hr
[3/100] BigCodeBench/2: 0P/4F | patches=1 | pass_rate=2/3 | 65/hr
[4/100] BigCodeBench/3: 4P/0F | patches=1 | pass_rate=3/4 | 69/hr


Step,Training Loss
5,3.040800


...

Step,Training Loss
5,2.092700


[90/100] BigCodeBench/89: 0P/4F | patches=28 | pass_rate=36/90 | 54/hr


Step,Training Loss
5,2.458600


Step,Training Loss
5,3.099300


[100/100] BigCodeBench/99: 0P/4F | patches=30 | pass_rate=40/100 | 54/hr

PATCH CREATION COMPLETE
Tasks processed: 100
Tasks with passes: 40
Patches created: 30
Time: 111.3 min
Bank stats: {'total': 30, 'high_q': 0, 'mid_q': 30, 'low_q': 0, 'mean_q': 0.5, 'total_memory_mb': 105.46875, 'avg_rank': 2.0}


In [ ]:
# Cell 4b: Verify patches are strong enough
import torch
import numpy as np

if len(patch_bank.patches) == 0:
    print('No patches created yet!')
else:
    print('=== PATCH STRENGTH CHECK ===')
    print()

    # Check 1: Weight magnitudes
    print('[1] Weight magnitudes:')
    for patch in patch_bank.patches[:3]:
        patch_bank.load_weights(patch)
        norms = []
        for layer, w in patch.lora_weights.items():
            nA = torch.norm(w['A']).item()
            nB = torch.norm(w['B']).item()
            norms.append(nA + nB)
        avg = sum(norms) / len(norms) if norms else 0
        first_key = list(patch.lora_weights.keys())[0]
        w = patch.lora_weights[first_key]
        print('  {}: |A|={:.4f} |B|={:.4f} avg_norm={:.4f}'.format(
            patch.patch_id[:40], torch.norm(w['A']).item(), torch.norm(w['B']).item(), avg))
        patch.unload_weights()

    # Check 2: Does output change?
    print()
    print('[2] Output difference (first 3 eval tasks):')
    for task in EVAL_TASKS[:3]:
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        emb = embedder.encode(prompt).tolist()
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

        torch.manual_seed(42)
        out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

        active = patch_bank.get_active_patches(emb, top_k=1)
        for p in active:
            patch_bank.load_weights(p)

        torch.manual_seed(42)
        try:
            with PatchedModel(base_model, active):
                out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)
        finally:
            for p in active:
                p.unload_weights()

        same = out_cold == out_patched
        if same:
            print('  {}: IDENTICAL (patch has zero effect)'.format(task['task_id']))
        else:
            cold_tokens = out_cold.split()
            patch_tokens = out_patched.split()
            diff = sum(1 for a, b in zip(cold_tokens, patch_tokens) if a != b)
            total = max(len(cold_tokens), len(patch_tokens))
            pct = diff / max(total, 1) * 100
            print('  {}: {:.0f}% tokens different'.format(task['task_id'], pct))

    # Check 3: Does patched output make sense?
    print()
    print('[3] Sample patched output:')
    task = EVAL_TASKS[0]
    prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
    emb = embedder.encode(prompt).tolist()
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
    active = patch_bank.get_active_patches(emb, top_k=1)
    for p in active:
        patch_bank.load_weights(p)
    try:
        with PatchedModel(base_model, active):
            out = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(out)
        has_def = 'def ' in code
        has_import = 'import ' in code
        print('  Valid Python?', has_def or has_import)
        print('  First 300 chars:', code[:300])
    finally:
        for p in active:
            p.unload_weights()

    print()
    if avg > 0.01:
        print('Patches have weight. Good.')
    else:
        print('WARNING: Weights too small. Increase steps or lr.')


In [ ]:
# Cell 4c: Check composition works (hook-based for 4-bit models)
import torch

task = EVAL_TASKS[0]
prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
emb = embedder.encode(prompt).tolist()
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

active = patch_bank.get_active_patches(emb, top_k=1)
for p in active:
    patch_bank.load_weights(p)
print('Active patches:', len(active))
print('Patch:', active[0].patch_id if active else 'none')

# Check 1: Greedy output difference
print()
print('=== Greedy (temp=0) ===')
torch.manual_seed(42)
out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

torch.manual_seed(42)
with PatchedModel(base_model, active):
    out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)

print('Cold first 100:', out_cold[:100])
print('Patched first 100:', out_patched[:100])
print('IDENTICAL:', out_cold == out_patched)

# Check 2: How many hooks were registered?
print()
print('=== Hook Count ===')
pm = PatchedModel(base_model, active)
pm.__enter__()
print('Hooks registered:', len(pm._hooks))
pm.__exit__(None, None, None)
print('Hooks removed:', len(pm._hooks) == 0)

# Check 3: Low-temp sampling
print()
print('=== Low-temp (0.01) ===')
torch.manual_seed(42)
out_cold_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

torch.manual_seed(42)
with PatchedModel(base_model, active):
    out_patched_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

print('IDENTICAL:', out_cold_lt == out_patched_lt)

if out_cold == out_patched and out_cold_lt == out_patched_lt:
    print('
Patch has ZERO effect. Check hook registration.')
elif out_cold == out_patched and out_cold_lt != out_patched_lt:
    print('
Patch works but too weak for greedy. Use sampling in eval.')
elif out_cold != out_patched:
    print('
Patch changes greedy output! It works.')

for p in active:
    p.unload_weights()


In [1]:
from cogmem.patches.bank import PatchBank
patch_bank = PatchBank("/notebooks/cogmem_patches")
patch_bank.load()
print(f"Patches: {len(patch_bank.patches)}")


Loaded 30 patches (0 promoted) from /notebooks/cogmem_patches
Patches: 30


In [5]:
# Cell 5: Evaluate patches on UNSEEN tasks (1040 tasks)
# Compare: cold (no patches) vs patched (with patches)
# Only eval a subset for speed — change EVAL_SIZE for full eval
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

EVAL_SIZE = 200  # first 200 of the 1040 unseen tasks
eval_subset = EVAL_TASKS[:EVAL_SIZE]
print("Evaluating on", EVAL_SIZE, "unseen tasks")
print("Patches in bank:", len(patch_bank.patches))

# --- Cold eval (no patches) ---
print()
print("--- COLD EVAL (base model, no patches) ---")
cold_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            cold_passed += 1
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print("  [{}/{}] cold: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, cold_passed, i + 1, cold_passed / (i + 1)))

cold_rate = cold_passed / max(EVAL_SIZE, 1)
print("Cold result:", cold_passed, "/", EVAL_SIZE, "({:.1%})".format(cold_rate))

# --- Patched eval (with cognitive patches) ---
print()
print("--- PATCHED EVAL (base model + patches per task) ---")
from cogmem.patches.compose import PatchedModel

patched_passed = 0
for i, task in enumerate(eval_subset):
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    # Get relevant patches for this task
    active_patches = patch_bank.get_active_patches(task_embedding, top_k=1)
    for p in active_patches:
        patch_bank.load_weights(p)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    try:
        with PatchedModel(base_model, active_patches):
            response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")
        if result["passed"]:
            patched_passed += 1
    except Exception:
        pass
    finally:
        for p in active_patches:
            p.unload_weights()

    if (i + 1) % 50 == 0:
        print("  [{}/{}] patched: {}/{} ({:.1%})".format(
            i + 1, EVAL_SIZE, patched_passed, i + 1, patched_passed / (i + 1)))

patched_rate = patched_passed / max(EVAL_SIZE, 1)
print("Patched result:", patched_passed, "/", EVAL_SIZE, "({:.1%})".format(patched_rate))


Evaluating on 200 unseen tasks
Patches in bank: 30

--- COLD EVAL (base model, no patches) ---
  [50/200] cold: 0/50 (0.0%)
  [100/200] cold: 0/100 (0.0%)
  [150/200] cold: 0/150 (0.0%)
  [200/200] cold: 0/200 (0.0%)
Cold result: 0 / 200 (0.0%)

--- PATCHED EVAL (base model + patches per task) ---


NameError: name 'embedder' is not defined

In [ ]:
# Cell 6: Results comparison
print("=" * 50)
print("COGNITIVE PATCHES EXPERIMENT RESULTS")
print("=" * 50)
print()
print("Patches created from first 100 tasks:", len(patch_bank.patches))
print("Evaluated on", EVAL_SIZE, "UNSEEN tasks (tasks 100-{})".format(100 + EVAL_SIZE))
print()
print("{:<20} {:>8} {:>8} {:>10}".format("Model", "Passed", "Total", "Rate"))
print("-" * 48)
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Cold (no patches)", cold_passed, EVAL_SIZE, cold_rate))
print("{:<20} {:>8} {:>8} {:>9.1%}".format("Patched", patched_passed, EVAL_SIZE, patched_rate))
print()
diff = patched_rate - cold_rate
if diff > 0.01:
    print("Patches IMPROVED by {:.1%} on unseen tasks!".format(diff))
    print("The architecture WORKS - patches transfer to new tasks.")
elif diff > -0.01:
    print("No significant difference.")
    print("Patches didn't help (yet). May need more patches or better contrasts.")
else:
    print("Patches HURT by {:.1%}.".format(abs(diff)))
    print("Composition may be interfering. Check patch quality.")

# Patch bank details
print()
print("Patch bank:")
stats = patch_bank.stats()
for k, v in stats.items():
    print("  {}: {}".format(k, v))


In [ ]:
# Cell 7: Inspect individual patches
# See what the patches learned
for patch in patch_bank.patches[:5]:
    print("Patch:", patch.patch_id)
    print("  Source:", patch.source_task_id)
    print("  Type:", patch.source_type)
    print("  Q:", patch.q_value, "visits:", patch.q_visits)
    print("  Description:", patch.description[:100])
    print("  Memory:", patch.memory_bytes(), "bytes")
    print("  Layers:", len(patch.lora_weights))
    print()
